### ERA5 1° daily data, Oct 1978 – Dec 1979

**Superseded.** E4 now reads `ERA5/day_1deg_arco/`, 24-hour means on the regular 1° grid built from public hourly data by `ERA5-monthly-1deg-ARCO.ipynb`. This notebook records how the earlier daily file was built; that file is in the Hugging Face revision `phase1-2026-09`.

A data-preparation step for E4: builds the ERA5 daily reference it reads, `ERA5/day_1deg/ERA5_daily_1deg_19781001-19791231.nc`. The ERA5 data on the DKRZ store has no daily files.

- **Source:** `gs://vcm-ml-intermediate/2026-03-19-era5-1deg-8layer-1940-2025.zarr`, the 6-hourly ERA5 dataset Ai2 prepared for training ACE. It was conservatively regridded from 0.25° onto a 180 × 360 Gaussian grid (F90), not the regular 1° grid of the `mon_1deg` files. This bucket is private, so outside users can't rerun this notebook. The output file in the published dataset is the public copy.
- **Method:** each daily value is the mean of the four 6-hourly values at 00, 06, 12 and 18 UTC, labelled with the day's 00 UTC time. Variables are renamed to their CMIP names, and the pressure-level variables are stacked on a `plev` dimension at the evaluation levels. `ts` is the merged sea-surface and skin temperature, and `zg` is the 500 hPa geopotential height.
- **Period:** 1978-10-01 to 1979-12-31, matching the models' first 15 months of daily output. E4 evaluates 1979.

Paths are relative to `AIMIP_DATA_ROOT`, as in the evaluation notebooks. The default, `../../../local_data`, is the repo's `local_data/` as seen from this directory. Running the notebook replaces the existing output file with a new one; hard links to the old file keep the old contents.

In [1]:
import os
import sys
from dataclasses import dataclass

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

sys.path.insert(0, os.path.abspath('..'))
from aimip_data_utils import EVALUATION_PRESSURE_LEVELS

In [2]:
ERA5_1DEG_ZARRPATH = 'gs://vcm-ml-intermediate/2026-03-19-era5-1deg-8layer-1940-2025.zarr'
SUBSET_PERIOD = ('1978-10-01', '1979-12-31')
DATA_ROOT = os.environ.get('AIMIP_DATA_ROOT', '../../../local_data')
OUTPUT_PATH = os.path.join(DATA_ROOT, 'ERA5/day_1deg/ERA5_daily_1deg_19781001-19791231.nc')
STEPS_PER_DAY = 4 # 6-hourly source

In [3]:
@dataclass
class PressureLevelVar:

    pressure_level_name_template: str
    levels_pa: list[int]

    def get_pressure_level_var(self, ds: xr.Dataset, pressure_dim_name: str='plev') -> xr.DataArray:
        pressure_levels_hpa = [int(level_pa / 100) for level_pa in self.levels_pa]
        pressure_level_varnames = [self.pressure_level_name_template.format(level_hpa=level_hpa) for level_hpa in pressure_levels_hpa]
        da_pressure_var = ds[pressure_level_varnames].to_array(dim=pressure_dim_name)
        return da_pressure_var.assign_coords({pressure_dim_name: self.levels_pa})

In [4]:
ds = xr.open_zarr(ERA5_1DEG_ZARRPATH, chunks={'time': 360})

/tmp/ipykernel_1573860/2955829944.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 360. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_zarr(ERA5_1DEG_ZARRPATH, chunks={'time': 360})


In [5]:
SINGLE_LEVEL_EVAL_VAR_NAME_MAPPING = {
    'huss': 'Q2m',
    'pr': 'PRATEsfc',
    'ps': 'PRESsfc',
    'psl': 'PRMSL',
    'tas': 'TMP2m',
    'ts': 'merged_sea_surface_and_skin_temperature',
    'uas': 'UGRD10m',
    'vas': 'VGRD10m',
    'zg': 'h500',
}

PRESSURE_LEVEL_EVAL_VAR_NAME_MAPPING = {
    'hus': PressureLevelVar('Q{level_hpa}', EVALUATION_PRESSURE_LEVELS),
    'ta': PressureLevelVar('TMP{level_hpa}', EVALUATION_PRESSURE_LEVELS),
    'ua': PressureLevelVar('UGRD{level_hpa}', EVALUATION_PRESSURE_LEVELS),
    'va': PressureLevelVar('VGRD{level_hpa}', EVALUATION_PRESSURE_LEVELS),
}

In [6]:
ds_time_subset = ds.sel(time=slice(*SUBSET_PERIOD))

ds_time_subset_eval_vars = xr.Dataset()

for cmip_name, era5_name in SINGLE_LEVEL_EVAL_VAR_NAME_MAPPING.items():
    ds_time_subset_eval_vars[cmip_name] = ds_time_subset[era5_name]

for cmip_name, era5_pressure_var in PRESSURE_LEVEL_EVAL_VAR_NAME_MAPPING.items():
    ds_time_subset_eval_vars[cmip_name] = era5_pressure_var.get_pressure_level_var(ds_time_subset)

In [7]:
# averaging blocks of 4 steps only gives daily means if the subset is unbroken 6-hourly data starting at 00 UTC
times = ds_time_subset_eval_vars.time
assert (np.diff(times.values) == np.timedelta64(6, 'h')).all()
assert times.dt.hour.values[0] == 0
assert times.size % STEPS_PER_DAY == 0, times.size

ds_time_subset_daily_eval_vars = ds_time_subset_eval_vars.coarsen({'time': STEPS_PER_DAY}, coord_func=np.min).mean()
ds_time_subset_daily_eval_vars.attrs = {
    'title': 'ERA5 daily means on a 180 x 360 Gaussian grid, 1978-10-01 to 1979-12-31',
    'source': f'{ERA5_1DEG_ZARRPATH} (6-hourly ERA5, conservatively regridded from 0.25 degree)',
    'history': (
        'Daily mean of the 00, 06, 12 and 18 UTC values, labelled with the 00 UTC time; variables renamed to CMIP names. '
        'Created by evaluations/notebooks/data_prep/ERA5-daily-1deg.ipynb.'
    ),
}

In [8]:
ds_time_subset_daily_eval_vars.info()

xarray.Dataset {
dimensions:
	time = 457 ;
	latitude = 180 ;
	longitude = 360 ;
	plev = 7 ;

variables:
	float32 huss(time, latitude, longitude) ;
		huss:units = kg/kg ;
		huss:long_name = 2m specific humidity ;
	float32 pr(time, latitude, longitude) ;
		pr:long_name = Surface precipitation rate ;
		pr:short_name = mtpr ;
		pr:units = kg/m**2/s ;
	float32 ps(time, latitude, longitude) ;
		ps:long_name = Surface pressure ;
		ps:short_name = sp ;
		ps:standard_name = surface_air_pressure ;
		ps:units = Pa ;
	float32 psl(time, latitude, longitude) ;
		psl:long_name = Mean sea level pressure ;
		psl:short_name = msl ;
		psl:standard_name = air_pressure_at_mean_sea_level ;
		psl:units = Pa ;
	float32 tas(time, latitude, longitude) ;
		tas:long_name = 2m air temperature ;
		tas:short_name = t2m ;
		tas:units = K ;
	float64 ts(time, latitude, longitude) ;
		ts:long_name = Merged sea surface and skin temperature ;
		ts:units = K ;
	float32 uas(time, latitude, longitude) ;
		uas:long_name = 10m

In [9]:
with ProgressBar():
    ds_time_subset_daily_eval_vars = ds_time_subset_daily_eval_vars.compute()

[########################################] | 100% Completed | 100.81 s


In [10]:
# write to a new file and rename it into place: writing directly would truncate the existing file's inode,
# changing every hard link to it (e.g. a publication staging tree), and leave a partial file on failure
tmp_path = OUTPUT_PATH + '.tmp'
ds_time_subset_daily_eval_vars.to_netcdf(tmp_path)
os.replace(tmp_path, OUTPUT_PATH)